In [223]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
df_train = pd.read_csv('cars_ml_top_brands_with_generation_code.csv')
df_train.head()



,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,condition,generation,generation_code
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,NaN,2016 - 2019 3 поколение рестайлинг (U5),U5
1,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,NaN,2018 - 2020 6 поколение рестайлинг (AD/ADA),AD/ADA
2,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,NaN,2010 - 2014 3 поколение (SL),SL
3,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,NaN,2013 - 2017 J150 рестайлинг,J150
4,Toyota,Avalon,2020,21500000,Астана,90000.0,Седан,3.5,бензин,Автомат,Передний привод,Слева,NaN,NaN,2018 - н.в. XX50,XX50


In [224]:
CURRENT_YEAR = 2026
import re


def process_data(df):
    df = df.copy()
    df['price'] = np.log1p(df['price'])
    

    #drop
    df = df.drop("generation", axis=1)
    df = df.drop("condition", axis=1)
    df = df.drop("color", axis=1)
    df = df.drop("city", axis=1)
    df = df.dropna(subset=["brand", "model", "fuel_type"])


    #fillna
    df['engine_volume_l'] = df['engine_volume_l'].fillna(df['engine_volume_l'].median())
    #df['has_gen_number'] = df['generation_number'].apply(lambda x: 0 if pd.isna(x) else 1)
    df['mileage_km'] = df['mileage_km'].fillna(0)
    df['is_new'] = df['mileage_km'].apply(lambda x: 1 if x == 0 else 0)
    counts = df["brand"].value_counts()

    brands_to_keep = counts[counts > 400].index.tolist()

    df = df[df["brand"].isin(brands_to_keep)]

    return df

df_train = process_data(df_train)
df_train.tail()



,brand,model,year,price,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,generation_code,is_new
4087,Toyota,Land Cruiser Prado,2004,16.705882,300000.0,Внедорожник,4.0,petrol,automatic,Полный привод,Слева,J120,0
4088,Mercedes-Benz,E 320,1999,15.274126,150000.0,Седан,3.2,petrol,automatic,Задний привод,Слева,W210/S210,0
4089,Kia,Optima,2003,14.457365,0.0,Седан,2.0,petrol,manual,Передний привод,Слева,GH,1
4090,ВАЗ,(Lada) Priora 2170,2013,13.997833,0.0,Седан,1.6,petrol,manual,Передний привод,Слева,unknown,1
4091,Kia,Rio,2020,15.573369,129000.0,Седан,1.6,petrol,manual,Передний привод,Слева,unknown,0


In [225]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 4090 entries, 0 to 4091
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   brand            4090 non-null   str    
 1   model            4090 non-null   str    
 2   year             4090 non-null   int64  
 3   price            4090 non-null   float64
 4   mileage_km       4090 non-null   float64
 5   body_type        4090 non-null   str    
 6   engine_volume_l  4090 non-null   float64
 7   fuel_type        4090 non-null   str    
 8   transmission     4090 non-null   str    
 9   drive_type       4090 non-null   str    
 10  steering_wheel   4090 non-null   str    
 11  generation_code  4090 non-null   str    
 12  is_new           4090 non-null   int64  
dtypes: float64(3), int64(2), str(8)
memory usage: 447.3 KB


In [236]:
import numpy as np
from catboost import CatBoostRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X,y = df_train.drop('price', axis=1), df_train['price']
cat_columns = [
    "model",
    'brand',
    "body_type",
    "fuel_type",
    "transmission",
    "drive_type",
    "steering_wheel",
    "generation_code"
    
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    early_stopping_rounds=300,
    verbose=500
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_columns,
    eval_set=(X_test, y_test),
    use_best_model=True
)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'R²: {r2:.4f}')  

0:	learn: 0.9494703	test: 0.9388406	best: 0.9388406 (0)	total: 4.75ms	remaining: 47.5s
500:	learn: 0.2394459	test: 0.2380497	best: 0.2380497 (500)	total: 827ms	remaining: 15.7s
1000:	learn: 0.2114537	test: 0.2327777	best: 0.2327692 (998)	total: 1.69s	remaining: 15.2s
1500:	learn: 0.1933477	test: 0.2315903	best: 0.2312465 (1314)	total: 2.5s	remaining: 14.2s
2000:	learn: 0.1791590	test: 0.2312136	best: 0.2309550 (1728)	total: 3.4s	remaining: 13.6s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.2309550414
bestIteration = 1728

Shrink model to first 1729 iterations.
MAE: 0.1596
MSE: 0.0533
R²: 0.9420


In [237]:
manual_car = pd.DataFrame([{    
    "model": "camry",
    'brand': "Toyota",
    "generation_code":"V55",
    "year": 2014,
    "body_type": "седан",
    "engine_volume_l": 2.5,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "передний привод",
    "steering_wheel": "Слева",
    "mileage_km": 154000,
    "is_new": 0,
}])

manual_car = manual_car.reindex(columns=X_train.columns, fill_value=0)
for col in cat_columns:
    manual_car[col] = manual_car[col].astype(str).fillna("unknown")
pred_log = model.predict(manual_car)

pred_price = np.expm1(pred_log)

print(f"Predicted price: {pred_price[0]:,.0f} ₸")

Predicted price: 13,329,464 ₸


In [238]:
df_train['generation_code'].value_counts()

generation_code
unknown            1183
XV50                 94
XV70                 92
NX4                  80
J150                 73
                   ... 
R1/R2/XR10/XR20       1
G06/F96               1
G01                   1
GD/MS                 1
GH                    1
Name: count, Length: 273, dtype: int64

<class 'pandas.DataFrame'>
Index: 4090 entries, 0 to 7958
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  4090 non-null   str    
 1   model                  4090 non-null   str    
 2   year                   4090 non-null   int64  
 3   price                  4090 non-null   float64
 4   mileage_km             4090 non-null   float64
 5   body_type              4090 non-null   str    
 6   engine_volume_l        4090 non-null   float64
 7   fuel_type              4090 non-null   str    
 8   transmission           4090 non-null   str    
 9   drive_type             4090 non-null   str    
 10  steering_wheel         4090 non-null   str    
 11  generation_start_year  4090 non-null   float64
 12  generation_number      2135 non-null   float64
 13  is_new                 4090 non-null   int64  
dtypes: float64(5), int64(2), str(7)
memory usage: 479.3 KB
